Add the project root to Python's import path so local modules (src/...) can be imported easily.

In [ ]:
import sys
from pathlib import Path

project_path = Path.cwd().parent.parent

sys.path.append(str(project_path.resolve()))

Get the API Keys

In [ ]:
import os

HF_TOKEN = os.getenv("HF_TOKEN", "False")
WANDB_API_KEY = os.getenv("WANDB_API_KEY", "False")

Import dataset helpers, evaluation utilities, Hugging Face Transformers and PEFT components used for training and tokenization.

In [ ]:
from huggingface_hub import login

login(token=HF_TOKEN)

In [ ]:
import wandb

In [ ]:
import torch
from unsloth import FastLanguageModel
from transformers import DataCollatorForLanguageModeling
from trl import SFTTrainer, SFTConfig

from src.dataset.load_data_soda import SODADataLoader

Load Tokenizer and Model from HuggingFace

In [ ]:
max_seq_length = 2048
load_in_4bit = True
dtype = None
load_in_8bit = False
model_name = "meta-llama/Llama-3.2-1B-Instruct"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit, # QLoRA
    dtype = dtype,
    load_in_8bit = load_in_8bit,
)

Create the PEFT model using the base model and LoRA configuration

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none", 
    use_gradient_checkpointing = "unsloth",
)

Create the SODA dataset loader with simple filtering options and retrieve the dataset object.

In [ ]:
soda_dataset_obj = SODADataLoader(
    percent_of_all_splits=1,
    min_story_length=20,
    max_story_length=1000,
    join_dialogue_and_speakers=True,
)
soda_ds = soda_dataset_obj.dataset

In [ ]:
soda_dataset_obj.set_tokenizer(tokenizer)

In [ ]:
soda_ds_formatted = soda_ds.map(soda_dataset_obj.formatting_prompts, batched=True)

In [ ]:
def get_model_parameter_counts(model):
    """
    Calculates and returns the total parameters, trainable parameters, 
    and the percentage of trainable parameters.
    """
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    trainable_percentage = (trainable_params / total_params) * 100

    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "trainable_percentage": trainable_percentage
    }

WANDB and HF repo settings

In [ ]:
wandb_project = os.getenv("WANDB_PROJECT", "story2dialogue-soda-llama3.2-1B-qlora")
hf_repo_id = os.getenv("HF_REPO_ID", False)

Initialize Weights & Biases ([wandb](https://wandb.ai/)) logging

In [ ]:
wandb.init(
    project=wandb_project,
    config={
        "model_name": model_name,
        "dataset": "SODA",
        "FastLanguageModel/max_seq_length": max_seq_length,
        "FastLanguageModel/load_in_4bit": load_in_4bit, 
        "FastLanguageModel/dtype": dtype,
        "FastLanguageModel/load_in_8bit": load_in_8bit
    }
    | soda_dataset_obj.get_dataset_info(include_word_counts=True)
    | model.peft_config['default'].__dict__
    | get_model_parameter_counts(model)
)

Create a data collator to batch examples correctly for LLM training.

In [ ]:
collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

Set training arguments for the Hugging Face Trainer (epochs, batch size, save/eval strategies, etc.).

In [ ]:
training_args = SFTConfig(
    output_dir="./llama_results",
    run_name="story2dialogue-SODA-Llama-1B-QLoRA",
    report_to="wandb",

    max_seq_length=2048,
    dataset_text_field="text",
    completion_only_loss=True,

    num_train_epochs=1,
    learning_rate=2e-4,
    per_device_train_batch_size=32,
    gradient_accumulation_steps=1,

    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    weight_decay=0.01,
    warmup_steps=10,

    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    push_to_hub=False,
    remove_unused_columns=True,
)

Create the SFTTrainer by wiring model, tokenizer, datasets and data collator together.

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = soda_ds_formatted["train"],
    eval_dataset = soda_ds_formatted["validation"],
    
    dataset_text_field = "text",
    
    max_seq_length = max_seq_length,
    data_collator = collator,
    args = training_args,
)

Start training the model. This will run for the number of epochs set in the training arguments.

In [ ]:
trainer_stats = trainer.train()

Upload fine-tuned model to HuggingFace

In [ ]:
commit_info = trainer.model.push_to_hub(hf_repo_id)
tokenizer.push_to_hub(hf_repo_id)

In [ ]:
wandb.log({"huggingface_commit_id": commit_info.oid})

In [ ]:
wandb.finish()